In [5]:
import pandas as pd
import numpy as np
from datetime import timedelta

### 建立模擬資料

In [16]:
data = {
    "customer_id": [
        "A", "A", "A",
        "B", "B",
        "C", "C"
    ],
    "purchase_date": [
        "2023-07-10",
        "2023-09-20",
        "2023-11-15",
        "2023-05-01",
        "2023-08-01",
        "2023-09-25",
        "2023-12-20"
    ],
    "amount": [
        100, 150, 150,
        80, 80,
        120, 200
    ]
}

df_tx = pd.DataFrame(data)
df_tx["purchase_date"] = pd.to_datetime(df_tx["purchase_date"])
df_tx 


,customer_id,purchase_date,amount
0,A,2023-07-10,100
1,A,2023-09-20,150
2,A,2023-11-15,150
3,B,2023-05-01,80
4,B,2023-08-01,80
5,C,2023-09-25,120
6,C,2023-12-20,200


### 定義snapshot date 觀察窗、預測窗參數

In [21]:
L = 90 #觀察窗參數
H = 60 # 預測窗參數
snapshot_dates = pd.to_datetime(["2023-09-30", "2023-10-31"]) # 定義觀察截止日snapshot date為每個月月底

In [22]:
customers = df_tx["customer_id"].unique() # 單獨抓出每個客戶 共有三個人ABC
customers


array(['A', 'B', 'C'], dtype=object)

In [23]:
base_rows = [
    {"customer_id":c, "snapshot_date":sp}
    for c in customers
    for sp in snapshot_dates    
]
df_panel = pd.DataFrame(base_rows)
df_panel
# 針對每個customer 的每個月的snapshot_date都建立一個row
# 流失模型的面板資料
# 笛卡爾積 
# 對每一個顧客，再搭配每一個 snapshot_date，形成所有可能的組合。

,customer_id,snapshot_date
0,A,2023-09-30
1,A,2023-10-31
2,B,2023-09-30
3,B,2023-10-31
4,C,2023-09-30
5,C,2023-10-31


In [24]:
# ====== 3. 對每個 (customer_id, snapshot_date) 計算特徵與標籤 ======

# 計算每個 customer_id 每個 snapshot_date 的特徵值（觀察窗過去90天的RFM標籤）
# 未來60天有沒有流失(流失標籤)
# timedelta(days = n) 回傳日期相減物件
feature_rows = []

for _, row in df_panel.iterrows():
    # iterrows 對df每個row迭代，每迭代一次就回傳row的index 還有這個row的資料series
    cid = row["customer_id"] # 把series的customer_id放到cid變數
    snapshot = row["snapshot_date"] # 把series的snapshot_date放到snapshot變數
    
    # --- 3.1 觀察窗（lookback window）---
    obs_start = snapshot - timedelta(days=L-1)  # 定義觀察窗 往回看90天包含snapshot那天
    obs_end   = snapshot                        # 觀察窗到snapshot為止 snapshot-89天 ~ snapshot 為觀察窗
    
    
    # 從交易資料df_tx中抓資料出來
    # 抓出客戶資料跟 customer x snapshot 的df一樣的數據
    # 根據df_tx過濾條件是 某顧客購買時間在觀察窗內的資料 是df包含 (customer_id 、purchase date、 amount)
    tx_obs = df_tx[
        (df_tx["customer_id"] == cid) &
        (df_tx["purchase_date"] >= obs_start) &
        (df_tx["purchase_date"] <= obs_end)
    ]
    
    # R / F / M
    if len(tx_obs) > 0: # 如果在觀察窗內有資料時
        last_purchase = tx_obs["purchase_date"].max() # 直接抓出該顧客的最後購買日期
        R_days = (snapshot - last_purchase).days # 計算r值最後購買日期距離snapshot有幾天
        F_orders_90d = len(tx_obs) # 計算f值在該客戶總購買紀錄中有幾筆
        M_amount_90d = tx_obs["amount"].sum() # 計算m值該客戶在該區間總購買金額
    else:
        # 觀察窗內完全沒下單：R 設為觀察窗長度上限，代表「至少這麼久沒買」
        R_days = L # 至少90天沒有買觀察參數90
        F_orders_90d = 0 # 邏輯是完全沒有買所以0
        M_amount_90d = 0.0 # 邏輯是完全沒有買所以是0
    
    # --- 3.2 預測窗（prediction window）---
    pred_start = snapshot + timedelta(days=1)  # 預測窗開始等同於snapshot+1天開始
    pred_end   = snapshot + timedelta(days=H)  # 預測窗結束等同於snapshot+60天結束
    
    # 篩選購買資料表中，購買紀錄在預測窗的資料
    tx_pred = df_tx[
        (df_tx["customer_id"] == cid) &
        (df_tx["purchase_date"] > pred_start - timedelta(days=0)) &  # 嚴格 > snapshot
        (df_tx["purchase_date"] <= pred_end)
    ]
    
    label_churn_60d = 0 if len(tx_pred) > 0 else 1 # 如果在60天預測窗有買則標記成0 反之流失1
    
    # 把所有變數存到feature_rows 存成df
    feature_rows.append({
        "customer_id": cid,
        "snapshot_date": snapshot,
        "R_days": R_days,
        "F_orders_90d": F_orders_90d,
        "M_amount_90d": M_amount_90d,
        "label_churn_60d": label_churn_60d
    })

df_features = pd.DataFrame(feature_rows).sort_values(
    ["customer_id", "snapshot_date"]
).reset_index(drop=True)

df_features


,customer_id,snapshot_date,R_days,F_orders_90d,M_amount_90d,label_churn_60d
0,A,2023-09-30,10,2,250.0,0
1,A,2023-10-31,41,1,150.0,0
2,B,2023-09-30,60,1,80.0,1
3,B,2023-10-31,90,0,0.0,1
4,C,2023-09-30,5,1,120.0,1
5,C,2023-10-31,36,1,120.0,0
